# GraspDecoder3D: random-input test
Builds `GraspDecoder3D` on its own (random weights, no encoder) and feeds it small hard-coded random tensors.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("."))   # run from the repo root

import torch
from grasp_point_sam.model.grasp_decoder import GraspDecoder3D, GraspPredictions, GRASP_DEPTHS
from grasp_point_sam.model.transformer import TwoWayTransformer
from grasp_point_sam.model.mask_decoder import AuxInputs, MaskDecoder

torch.manual_seed(0)

## Sizes
- `B` scenes, `M` prompted objects per scene, `L` patches, `N` points, `K` prompt tokens, `D` embedding dim

In [2]:
B = 1     # batch size (scenes)
M = 2     # prompted instances per scene
L = 32    # number of patches (encoder output tokens)
N = 500   # number of points
K = 1     # sparse prompt tokens per instance (1 click)
D = 256   # transformer_dim (must match config)

## Hard-coded random inputs
In the full model these come from the PointSAM encoder and prompt encoder. They're already expanded to `B*M` here, the same way `grasp_point_sam.py` does before calling the decoder.

In [3]:
# Point cloud: xyz in [-1, 1] and rgb in [0, 1]
coords   = torch.rand(B, N, 3) * 2 - 1          # [B, N, 3]
features = torch.rand(B, N, 3)                  # [B, N, 3]

# Patch centers: a random subset of the points
centers  = coords[:, torch.randperm(N)[:L]]     # [B, L, 3]

aux_inputs = AuxInputs(coords=coords, features=features, centers=centers)
# interp_index / interp_weight are left None -> the decoder computes them

# Encoder outputs (already repeated per instance)
pc_embeddings = torch.randn(B * M, L, D)        # [B*M, L, D]
pc_pe         = torch.randn(B * M, L, D)        # [B*M, L, D]

# Prompt embeddings
sparse_prompt_embeddings = torch.randn(B * M, K, D)   # [B*M, K, D]  (point clicks)
dense_prompt_embeddings  = torch.zeros(B * M, L, D)   # [B*M, L, D]  (no mask prompt)

for name, t in [("coords", coords), ("centers", centers), ("pc_embeddings", pc_embeddings),
                ("pc_pe", pc_pe), ("sparse", sparse_prompt_embeddings), ("dense", dense_prompt_embeddings)]:
    print(f"{name:15s} {tuple(t.shape)}")

coords          (1, 500, 3)
centers         (1, 32, 3)
pc_embeddings   (2, 32, 256)
pc_pe           (2, 32, 256)
sparse          (2, 1, 256)
dense           (2, 32, 256)


## Build the decoder (same settings as `configs/graspnet.yaml`)

In [ ]:
# PointSAM's decoder (random weights here; build_model loads the pretrained ones).
# The grasp decoder reuses its transformer, iou/mask tokens and mask heads.
mask_decoder = MaskDecoder(
    transformer_dim=D,
    transformer=TwoWayTransformer(depth=2, embedding_dim=D, num_heads=8, mlp_dim=2048),
)

decoder = GraspDecoder3D(
    mask_decoder=mask_decoder,
    transformer_dim=D,
    joint_mask_grasp=True,
    num_grasp_rot_tokens=6,
    num_depth_bins=len(GRASP_DEPTHS),
).eval()

print(f"params: {sum(p.numel() for p in decoder.parameters()) / 1e6:.2f} M")
print("transformer shared:", decoder.transformer is mask_decoder.transformer)

## Forward pass

In [14]:
output_tokens = torch.cat([
    decoder.iou_token.weight,
    decoder.mask_tokens.weight,
    decoder.grasp_quality_token.weight,
    decoder.grasp_rotation_tokens.weight,
    decoder.grasp_width_token.weight ], dim = 0)

output_tokens


tensor([[-0.5726, -0.6571, -0.0700,  ...,  0.0163,  0.9490, -0.3448],
        [ 2.0652, -2.3334, -1.1102,  ...,  1.1456, -0.5920,  0.0058],
        [-0.3532,  1.3941,  0.0060,  ..., -0.1242, -1.8702, -0.0262],
        ...,
        [ 0.1968,  1.7189,  0.0593,  ..., -0.4258,  1.0881, -1.2752],
        [ 0.3320,  0.0228,  1.0393,  ..., -1.0711,  0.5921, -0.3345],
        [ 0.5506, -1.8591, -0.3808,  ...,  1.7330, -0.9133,  0.9478]],
       grad_fn=<CatBackward0>)

In [15]:
output_tokens.shape

torch.Size([13, 256])

In [21]:
num_base_tokens = output_tokens.shape[0]
output_tokens = torch.cat([output_tokens, decoder.grasp_depth_tokens.weight], dim=0)

batch_size = sparse_prompt_embeddings.shape[0]
output_tokens = output_tokens.unsqueeze(0).expand(batch_size, -1, -1)
tokens = torch.cat([output_tokens, sparse_prompt_embeddings], dim=1)


In [22]:
output_tokens.shape

torch.Size([2, 21, 256])

In [20]:
sparse_prompt_embeddings.shape

torch.Size([2, 1, 256])

In [23]:
tokens.shape

torch.Size([2, 22, 256])

In [ ]:
with torch.no_grad():
    out = decoder(
        pc_embeddings, pc_pe,
        sparse_prompt_embeddings, dense_prompt_embeddings,
        aux_inputs, multimask_output=True,
    )

for name, v in vars(out).items():
    print(f"{name:15s} {None if v is None else tuple(v.shape)}")

grasp_quality   (2, 500)
grasp_rotation  (2, 500, 6)
grasp_width     (2, 500, 1)
masks           (2, 3, 500)
iou_preds       (2, 3)
grasp_depth     (2, 500, 4)


In [8]:
type(out)

grasp_point_sam.model.grasp_decoder.GraspPredictions

In [10]:
out.grasp_rotation.shape

torch.Size([2, 500, 6])

In [11]:
import torch.nn as nn
x = nn.Embedding(5, 3)
x.weight.data

tensor([[-0.3828,  0.7185,  0.4909],
        [ 0.3487,  0.6344,  0.2607],
        [-0.1639, -2.1321,  0.4882],
        [-0.0875,  1.5125, -1.2110],
        [-0.3347, -0.2644, -0.5916]])

In [12]:
x.weight.data.shape

torch.Size([5, 3])